In [11]:
!pip install hmmlearn


import pandas as pd
import os
import numpy as np
import joblib
from hmmlearn.hmm import GaussianHMM

# ==========================================
# STEP 1: LOAD DATASET
# ==========================================

url = "https://raw.githubusercontent.com/theshreyaaa/Dynamic-price-optimization-for-electric-vehicle-EV-technology/main/hourly_features_with_states.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 observations:")
print(df.head())


# ==========================================
# STEP 2: VERIFY DATASET
# ==========================================

df["timestamp"] = pd.to_datetime(df["timestamp"])

# Ensure chronological ordering
df = df.sort_values("timestamp").reset_index(drop=True)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTotal observations:", len(df))


# ==========================================
# STEP 3: OPERATING REGIME DISTRIBUTION
# ==========================================

state_counts = df["state_name"].value_counts()

print("\nNumber of hours in each operating regime:")
print(state_counts)

print("\nPercentage distribution:")
print((state_counts / len(df) * 100).round(2))


# ==========================================
# STEP 4: STATE-WISE FEATURE ANALYSIS
# ==========================================

state_analysis = df.groupby("state_name").agg({
    "occupancy_rate": "mean",
    "transformer_load_ratio": "mean"
})

print("\nAverage values for each operating regime:")
print(state_analysis)


# ==========================================
# STEP 5: TRANSITION COUNTS
# ==========================================

current_states = df["state"].iloc[:-1]
next_states = df["state"].shift(-1).iloc[:-1]

state_order = [0, 1, 2]

transition_counts = pd.crosstab(
    current_states,
    next_states
).reindex(
    index=state_order,
    columns=state_order,
    fill_value=0
)

print("\nTransition Counts:")
print(transition_counts)

print("\nTotal transitions:", transition_counts.values.sum())


# ==========================================
# STEP 6: TRANSITION PROBABILITY MATRIX
# ==========================================

transition_matrix = transition_counts.div(
    transition_counts.sum(axis=1),
    axis=0
)

print("\nTransition Probability Matrix:")
print(transition_matrix.round(6))


# ==========================================
# STEP 7: VALIDATION
# ==========================================

print("\nRow sums of transition matrix:")
print(transition_matrix.sum(axis=1))

print("\nValidation:")

if transition_matrix.shape == (3, 3):
    print("PASS: Three operating states verified.")

if transition_counts.values.sum() == len(df) - 1:
    print("PASS: Total transitions verified.")

if all(abs(transition_matrix.sum(axis=1) - 1) < 1e-9):
    print("PASS: Every transition probability row sums to 1.")


# ==========================================
# STEP 8: TRAIN HMM MODEL
# ==========================================

# Load original training dataset
# Load original training dataset from GitHub
training_url = "https://raw.githubusercontent.com/theshreyaaa/Dynamic-price-optimization-for-electric-vehicle-EV-technology/main/ev_charging_hourly_features.csv"

training_df = pd.read_csv(training_url)

print("Original training dataset loaded successfully!")
print("Dataset shape:", training_df.shape)
print("Columns:", training_df.columns.tolist())

X = training_df[
    ["occupancy_rate", "transformer_load_ratio"]
].values

# Train production model (K = 3)
model = GaussianHMM(
    n_components=3,
    covariance_type="full",
    n_iter=200,
    random_state=42
)

model.fit(X)


# ==========================================
# STEP 9: EXTRACT HMM PARAMETERS
# ==========================================

print("\nOriginal Transition Matrix:")
print(np.round(model.transmat_, 3))

print("\nEmission means [occupancy, load_ratio]:")
print(np.round(model.means_, 4))

print("\nCovariance matrices:")
print(np.round(model.covars_, 5))

print("\nStarting probabilities:")
print(np.round(model.startprob_, 3))


# ==========================================
# STEP 10: DECODE STATES (VITERBI)
# ==========================================

raw_states = model.predict(X)

# Sort states by average transformer load
# Lowest = Surplus
# Middle = Nominal
# Highest = Peak Stress

order = np.argsort(model.means_[:, 1])

rank = np.empty_like(order)
rank[order] = np.arange(3)

named_states = rank[raw_states]


# ==========================================
# REORDER HMM PARAMETERS
# ==========================================

# Reorder transition matrix according to named states
ordered_transmat = model.transmat_[
    np.ix_(order, order)
]

print("\nOrdered HMM Transition Matrix:")
print(np.round(ordered_transmat, 6))

# Initial probabilities in named state order
ordered_startprob = model.startprob_[order]

print("\nOrdered Initial State Probabilities:")
print(np.round(ordered_startprob, 6))

# Emission means in named state order
ordered_means = model.means_[order]

print("\nOrdered Emission Means [occupancy, load_ratio]:")
print(np.round(ordered_means, 6))

# Covariance matrices in named state order
ordered_covars = model.covars_[order]

print("\nOrdered Covariance Matrices:")
print(np.round(ordered_covars, 6))

# ==========================================
# HMM PARAMETER VALIDATION
# ==========================================

print("\n========== PHASE 4.1.3 ==========")
print("HMM PARAMETER VALIDATION")

# 1. Validate transition matrix
print("\nTransition matrix row sums:")
print(ordered_transmat.sum(axis=1))

# 2. Validate initial probabilities
print("\nInitial probability sum:")
print(ordered_startprob.sum())

# 3. Validate covariance matrix symmetry
print("\nCovariance matrix symmetry:")

for k in range(3):
    print(
        f"{labels[k]}:",
        np.allclose(
            ordered_covars[k],
            ordered_covars[k].T
        )
    )

# 4. Validate covariance matrix positive definiteness
print("\nCovariance matrix eigenvalues:")

for k in range(3):
    eigenvalues = np.linalg.eigvalsh(ordered_covars[k])

    print(f"{labels[k]}:", eigenvalues)

    assert np.all(eigenvalues > 0), (
        f"{labels[k]} covariance matrix is not positive definite"
    )

# 5. Final assertions
assert np.allclose(
    ordered_transmat.sum(axis=1), 1
)

assert np.isclose(
    ordered_startprob.sum(), 1
)

assert ordered_transmat.shape == (3, 3)
assert ordered_startprob.shape == (3,)
assert ordered_means.shape == (3, 2)
assert ordered_covars.shape == (3, 2, 2)

print("\nALL HMM PARAMETER VALIDATIONS PASSED!")

# State labels
labels = {
    0: "Surplus",
    1: "Nominal",
    2: "Peak Stress"
}

print("\nHours spent in each state:")

for k in range(3):
    print(
        f"  {labels[k]}: "
        f"{(named_states == k).sum()} hours"
    )


# ==========================================
# STEP 11: PERSISTENCE
# ==========================================

print("\nPersistence (diagonal of transition matrix):")

for k in range(3):
    print(
        f"  {labels[k]}: "
        f"{ordered_transmat[k, k]:.3f}"
    )


# ==========================================
# EMPIRICAL TRANSITION MATRIX
# ==========================================

print("\n========== PHASE 4.1.2 ==========")
print("EMPIRICAL TRANSITION PROBABILITY MATRIX")

# Use consecutive decoded states
previous_states = named_states[:-1]
next_states = named_states[1:]

# Calculate transition counts
empirical_counts = pd.crosstab(
    pd.Series(previous_states, name="Current State"),
    pd.Series(next_states, name="Next State")
)

# Ensure all three states appear
empirical_counts = empirical_counts.reindex(
    index=[0, 1, 2],
    columns=[0, 1, 2],
    fill_value=0
)

# Add readable state names
state_names = ["Surplus", "Nominal", "Peak Stress"]

empirical_counts.index = state_names
empirical_counts.columns = state_names

print("\nEmpirical Transition Counts:")
print(empirical_counts)

# Calculate transition probabilities row-wise
row_totals = empirical_counts.sum(axis=1)

empirical_probabilities = empirical_counts.div(
    row_totals.replace(0, np.nan),
    axis=0
).fillna(0)

print("\nEmpirical Transition Probability Matrix:")
print(empirical_probabilities.round(6))

# Verify total number of transitions
print("\nTotal transitions:", empirical_counts.values.sum())

# Verify row sums
print("\nTransition Probability Row Sums:")
print(empirical_probabilities.sum(axis=1).round(6))


# ==========================================
# STEP 12: SAVE OUTPUTS
# ==========================================

# Create folder before saving any files
os.makedirs("trained_models", exist_ok=True)

# Save empirical transition matrices
empirical_counts.to_csv(
    "trained_models/transition_counts.csv"
)

empirical_probabilities.to_csv(
    "trained_models/transition_probability_matrix.csv"
)

# Save trained HMM model
joblib.dump(
    model,
    "trained_models/gaussian_hmm_v1.pkl"
)

# Save decoded states
out = training_df.copy()

out["raw_state"] = raw_states
out["state"] = named_states
out["state_name"] = out["state"].map(labels)

out.to_csv(
    "hourly_features_with_states.csv",
    index=False
)

print("\nAll transition matrices saved successfully.")
print("HMM model and states file saved successfully.")


Dataset loaded successfully!
Dataset shape: (8760, 12)

Column names:
['timestamp', 'occupancy_rate', 'transformer_load_ratio', 'local_hour', 'local_dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'base_tou_price', 'raw_state', 'state', 'state_name']

First 5 observations:
                   timestamp  occupancy_rate  transformer_load_ratio  \
0  2019-01-01 00:00:00+00:00        0.000000                0.000000   
1  2019-01-01 01:00:00+00:00        0.000000                0.000000   
2  2019-01-01 02:00:00+00:00        0.000000                0.000000   
3  2019-01-01 03:00:00+00:00        0.018519                0.009825   
4  2019-01-01 04:00:00+00:00        0.018519                0.041562   

   local_hour  local_dayofweek  is_weekend  hour_sin      hour_cos  \
0          16                0           0 -0.866025 -5.000000e-01   
1          17                0           0 -0.965926 -2.588190e-01   
2          18                0           0 -1.000000 -1.836970e-16   
3          